# Lab 2 - ChipWhisperer's Python API

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY**: *In this lab, you'll learn how to use ChipWhisperer's Python API to connect to your ChipWhisperer hardware and change its settings. We'll also cover how to build firmware for your target microcontroller, how to capture power traces, and how to communicate with target devices.*

*All the API calls we'll be using here are documented on the [ChipWhisperer readthedocs page](https://chipwhisperer.readthedocs.io/en/sca101/scope-api.html), so feel free to open it in another tab and follow along there as well.*

**LEARNING OUTCOMES:**
* Setting up your ChipWhisperer Hardware
* Using the ChipWhisperer Python API to connect to your hardware
* Communication with the target
* Capturing a power trace

## Prerequisites

Hold up - before continuing, ensure you have done the following:

* ☑ Validated your setup using the [Connection Test](../Test%20Connection.ipynb) notebook.
* ☑ Run through [Lab 1 - Introduction to Jupyter](../Lab%201%20-%20Introduction%20to%20Jupyter/Lab.ipynb)

## Physical Setup

### CW-Husky with CW313/SAM4S

The only connections you need to make are with the 20-pin connector and the two SMA cables. The only settings to confirm are that JP2 is set to T-GPIO4 and that JP1 and JP4 are connected:

<img src="img/cwhusky.jpg" alt="CW-Husky Plugged In" width=800>

**Note that the two SMA connectors on the CW313 board connect to the same spot, so it doesn't matter which you connect to on that side. For SCA101, the Pos port on your Husky must be connected. Crowbar is not necessary, but may be left connected.**

## What is ChipWhisperer's Python API?

ChipWhisperer's Python API is a Python library that allows you to interact with your ChipWhisperer hardware. Among other things, it allows you to connect to your ChipWhisperer Husky, change its capture settings, program your SAM4S target board, and do SimpleSerial UART communications!

## Connecting to ChipWhisperer

Now that your hardware is all setup, we can now learn how to connect to it. We can connect to the ChipWhisperer with:

In [ ]:
import chipwhisperer as cw
scope = cw.scope()

Hopefully that block worked correctly. If not, try running through our [common connection errors page](https://chipwhisperer.readthedocs.io/en/sca101/cw_tips_tricks/common_errors/libusb_error_busy.html) on our documentation website.

Continuing on, your ChipWhisperer Husky is now connected, but its settings haven't been initialized. There's some defaults, but
they aren't suitable for capturing power traces. Luckily we've got a simple method call that will setup some sane defaults for you:

In [ ]:
scope.default_setup()

It even tells you what settings it changed. Most importantly, it:

* Sets the scope gain to 45dB
* Sets the scope to capture 5000 samples
* Sets the scope offset to 0 (aka it will begin capturing as soon as it is triggered)
* Sets the scope trigger to rising edge
* Outputs a 7.37MHz clock to the target on HS2
* Clocks the scope ADC at 4\*7.37MHz. Note that this is *synchronous* to the target clock on HS2
* Assigns GPIO1 as serial RX
* Assigns GPIO2 as serial TX

Basically, it just setup necessary IO pins, a clock for the target, and some sane capture settings.

One nice feature of our API is that you can simply `print()` our scope object to see its settings:

In [ ]:
print(scope)

All of these settings correspond directly to their API calls. For example, if we want to change how many samples we capture, if we go to the `adc` part of the printed string, we can see that `samples` are currently set to 5000.

Thus, we can change that by changing `scope.adc.samples`:

In [ ]:
scope.adc.samples = 4000

You won't need to mess around too much with these settings during this course, but being able to modify these things is very important for extending beyond SCA101.

## Setting Up Target Communications

Setting up target communications is similarly simple, just call `cw.target(scope)`:

In [ ]:
target = cw.target(scope, cw.targets.SimpleSerial) #cw.targets.SimpleSerial can be omitted

This creates a target object which we can use to send serial messages to our SAM4S (once it's programmed).

For this course, we're going to be using SimpleSerial v1.1, which, as the name suggests is a fairly simple serial protocol. We've got [detailed documentation on this protocol](https://chipwhisperer.readthedocs.io/en/sca101/simpleserial.html), but in general communication looks like this:

1. We send a packet to the target. This packet includes an ASCII character indicating a command to run and some ASCII encoded data
2. If the target has a matching command registered, it passes the sent data to an associated callback function
3. The target *may* respond with its own packet
4. The target *always* repsonds with an acknowledgement packet

An example you'll be seeing a lot of in this course is getting the target to encrypt some data using AES for us. This looks like:

1. We send a `'p'` packet with 16 bytes of data (AES always encrypts 16 bytes of data at a time)
2. The target receives the `'p'` command, then calls an AES encryption function, passing the 16 bytes of data we sent to it
3. The target sends 16 bytes of encrypted data back to us
4. The target respons with an acknowledgement packet

## Building and Uploading Firmware

The next step in attacking a target is to get some firmware built and uploaded onto it. If you've followed our installation instructions, you should be all set up with an arm compiler to build firmware for the SAM4S.

Let's take a look at the firmware we'll be building - simpleserial-base. You can find this firmware in the main chipwhisperer repo at `firmware/mcu/simpleserial-base/simpleserial-base.c`.

There's a few things going on, mostly setup, but you should draw your attention to the following function:

```C
uint8_t get_pt(uint8_t* pt, uint8_t len)
{
	/**********************************
	* Start user-specific code here. */
	trigger_high();

	//16 hex bytes held in 'pt' were sent
	//from the computer. Store your response
	//back into 'pt', which will send 16 bytes
	//back to computer. Can ignore of course if
	//not needed

	trigger_low();
	/* End user-specific code here. *
	********************************/
	simpleserial_put('r', 16, pt);
	return 0x00;
}
```

and the following function call:

```C
simpleserial_addcmd('p', 16, get_pt);
```

As you can see, the target firmware is setup to look for a `'p'` command with 16 bytes of data. When it receives this command, it calls `get_pt()`, passing the data we send in the first parameter, `pt`. This function doesn't really do anything, it just toggles the target's trigger pin, then sends the data back to us, but this is enough to show off the basics of communication.

Let's build this firmware, upload it, and test it out! Firmware must be built on the command line. Luckily, thanks to Jupyter, we can run a command within a notebook as follows:

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-base/
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_1_1

You should see some messages printed out saying various files were compiled/linked/etc, then something at the end like so:

```
+--------------------------------------------------------
+ Built for platform CW-Husky Microchip SAM4S with:
+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = AES128C
+--------------------------------------------------------
```

As you can see we built firmware for the CW Husky's SAM4S. You can also use this build system to build firmware for various other target boards by changing the `PLATFORM` field in the `make` command, but we won't be doing that in this course.

Now, let's program this firmware onto our target board. We've got a built in bootloader for the SAM4S target, so we can program directly from the API:

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

That hopefully will have completed without error.

Next we'll be learning how to capture power traces and communicate with the target.

## Communication with the Target

Communication with targets, which is done through the `SimpleSerial target` object we got earlier, is grouped into two categories:

1. Raw serial via `target.read()`, `target.write()`, `target.flush()`, etc. 

1. SimpleSerial commands via `target.simpleserial_read()`, `target.simpleserial_write()`, `target.simpleserial_wait_ack()`, etc.

The firmware we uploaded uses the [simpleserial protocol](https://chipwhisperer.readthedocs.io/en/sca101/simpleserial.html), so we'll start off with simpleserial. Later, we'll use the raw serial commands to send the same messages.

If you check the simpleserial-base firmware (`simpleserial-base.c`) you'll find that for the simpleserial `'p'` command, the target will echo back the command. Let's try that out now:

In [ ]:
msg = bytearray([0]*16) #simpleserial uses bytearrays
target.simpleserial_write('p', msg)

Let's check if we got a response:

In [ ]:
print(target.simpleserial_read('r', 16))

If you go back to your firmware file, you can see that the target also has a `'k'` command that also takes 16 bytes.

**Try sending a `'k'` command in the block below:**

That command doesn't return anything to us, but it should ack and give us an acknowledgement which we can check with `simpleserial_wait_ack()`:

In [ ]:
assert target.simpleserial_wait_ack() == 0, "Target did not ack. Did you send the 'k' command?" #should return 0

Simpleserial messages generally take the form:

```python
command_character + ascii_encoded_bytes + '\n'
```

For our first command, `command_character='p'` and `ascii_encoded_bytes="00"*16` (keep in mind this isn't a binary `0x00`, it's ASCII `"00"`, which has a binary value of `0x3030`). 

**Try resending the `'p'` command from earlier using `target.write()`**:

In [ ]:
target.write('p' + ) #fill in the rest here

A simple `target.read()` will return all the characters that have been sent back from the target so far. Let's see what the device returned to us:

In [ ]:
recv_msg = ""

In [ ]:
recv_msg += target.read() #you might have to run this block a few times to get the full message
print(recv_msg)

In [ ]:
assert recv_msg != "", "Received message blank, did you send the 'p' command?"

The simpleserial commands are usually sufficient for taking to simpleserial firmware, but you'll need the raw serial commands for some of the other labs.

## SimpleSerial 2

If you read our SimpleSerial documentation, you'll probably saw a section on SimpleSerial 2. SimpleSerial 2 is a new protocol with a number of advantages over the old protcol:

1. Data is unencoded binary (except for byte stuffing) instead of ACII encoded, meaning far more data can be sent per packet (just under double)
1. It's got a command and subcommand field that get passed to the target callback functions, meaning more can be done per packet. For example, the simpleserial-aes firmware has been modified to allow plaintext, key, and/or mask to be set with a single packet.
1. It's got a length field, meaning the same target commands can take different length packets depending on the situation. This, for example, allows both plaintext and key (or only one) to be send with the same command in simpleserial-aes.
1. It's got an 8-bit CRC (0xA6) for data integrety.
1. Frames are consistant overhead byte stuffed (COBS). This makes it easy to reset communication (done simply by sending two 0x00 bytes) and helps catch malformed length bytes.

It is, however, a more complicated protocol. You can use it by compiling firmware with `SS_VER=SS_VER_2_1` and using `cw.targets.SimpleSerial2` instead of `cw.targets.SimpleSerial`. For SCA101, however, we recommend sticking with SimpleSerial 1.

## Capturing Traces

Now that the target's programmed and we know how to communicate with it, let's start recording some power traces! To capture a trace:

1. Arm the ChipWhisperer with `scope.arm()`. It will begin capturing as soon as it is triggered (which in our case is a rising edge on `gpio4`).
1. `scope.capture()` will read back the captured power trace, blocking until either ChipWhisperer is done recording, or the scope times out. Note that the error return will tell you whether or not the scope timed out. It does not return the captured scope data.
1. You can read back the captured power trace with `scope.get_last_trace()`.

`simpleserial_base` will trigger the ChipWhisperer when we send the `'p'` command. Try capturing a trace now:

In [ ]:
scope.arm()
target.simpleserial_write('p', msg)
## fill in the rest...

ChipWhisperer also has a `capture_trace()` convience function that:

1. Optionally sends the `'k'` command
1. Arms the scope
1. Sends the `'p'` command
1. Captures the trace
1. Reads the return `'r'` message
1. Returns a `TraceContainer` class that groups the trace data, `'p'` message, the `'r'` message, and the `'k'` message

It isn't always the best option to use, but it's usually sufficient for SimpleSerial based firmware, so we'll use it in most future labs.

## Conclusion

And that's it! You should be all ready to continue on to the next lab.

We've glossed over some stuff here, so consult the [API documentation](https://chipwhisperer.readthedocs.io/en/sca101/scope-api.html) or ask on our [forums](https://forum.newae.com) if you get stuck.

We've also got a [Cheat Sheet Notebook](../ChipWhisperer%20Cheat%20Sheet.ipynb) focusing on common code blocks useful when interacting with ChipWhisperer's API.

## Disconnecting From Hardware

Before moving on to the next lab, you must disconnect your ChipWhisperer-Husky by running the code block below.

This is because only one notebook can be connected to a USB device at a time, so if you don't run this block, you may **run into errors in the next lab.**

In [ ]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>